# ch11. 특이값분해(SVD) — ML에서 가장 중요한 분해 🌟

웹 챕터 11에서 배운 내용을 NumPy 로 직접 확인합니다.

**이 노트북에서 하는 것**
1. 행렬이 **단위원을 타원**으로 보내는 모습 + 특이값 = 타원 반지름 확인
2. `np.linalg.svd` 로 **A = UΣVᵀ** 분해하고 다시 조립해 검증
3. **이미지 저랭크 복원**: rank 1 / 5 / 20 근사로 압축 효과를 눈으로 확인
4. 특이값 크기 분포(에너지) 그래프

> 용어 복습. **특이값 σ**: 각 방향의 확대 배율(항상 0 이상, 큰 순서로).
> **U, V**: 회전(정규직교). **저랭크 근사**: σ 큰 조각 k 개만 남기기, 곧 최선의 rank k 근사.

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [1]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch11')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (그래프의 한글이 깨지지 않게)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('numpy', np.__version__, '준비 완료')

## 1. 단위원 → 타원, 특이값 = 타원의 반지름

단위원 위의 점들을 행렬 $A$ 로 변환하면 **항상 타원**이 됩니다.
`np.linalg.svd` 가 돌려주는 특이값이 정말 타원의 두 반지름인지 그려서 확인해 봐요.

In [ ]:
A = np.array([[1.5, 0.5],
              [0.0, 1.0]])

U, S, Vt = np.linalg.svd(A)
print('특이값 σ =', S.round(3))

# 단위원 위 점들을 A 로 변환
t = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(t), np.sin(t)])   # 2 x 200
ellipse = A @ circle

# 타원의 두 축 = σ_i * u_i  (U 의 열 방향으로 길이 σ)
axis1 = S[0] * U[:, 0]
axis2 = S[1] * U[:, 1]

plt.figure(figsize=(5, 5))
plt.plot(circle[0], circle[1], '--', color='#94a3b8', label='단위원 (변환 전)')
plt.plot(ellipse[0], ellipse[1], color='#6366f1', lw=2, label='A x 단위원 = 타원')
plt.plot([0, axis1[0]], [0, axis1[1]], color='#10b981', lw=3, label=f'축1 (길이 σ1={S[0]:.2f})')
plt.plot([0, axis2[0]], [0, axis2[1]], color='#f59e0b', lw=3, label=f'축2 (길이 σ2={S[1]:.2f})')
plt.axhline(0, color='gray', lw=0.5)
plt.axvline(0, color='gray', lw=0.5)
plt.xlim(-2.2, 2.2); plt.ylim(-2.2, 2.2)
plt.gca().set_aspect('equal')
plt.title('행렬은 단위원을 타원으로 보낸다')
plt.legend(loc='lower right', fontsize=9)
plt.show()

print(f'σ1 x σ2 = {S[0]*S[1]:.3f}  vs  |det A| = {abs(np.linalg.det(A)):.3f}  <- 같다! (ch6 넓이 배율)')

## 2. A = UΣVᵀ 분해하고 다시 조립하기

세 조각(회전 · 확대 · 회전)을 도로 곱하면 정확히 A 가 나오는지,
그리고 U, V 가 정말 **정규직교**(ch10)인지 확인합니다.

In [ ]:
Sigma = np.diag(S)                 # 특이값을 대각행렬로
A_rebuilt = U @ Sigma @ Vt         # 회전 x 확대 x 회전

print('U =\n', U.round(3))
print('Σ =\n', Sigma.round(3))
print('Vᵀ =\n', Vt.round(3))
print()
print('U @ Σ @ Vᵀ 가 A 와 같은가?', np.allclose(A, A_rebuilt))
print('UᵀU = I (정규직교)?      ', np.allclose(U.T @ U, np.eye(2)))
print('VᵀV = I (정규직교)?      ', np.allclose(Vt @ Vt.T, np.eye(2)))

## 3. 이미지 저랭크 복원: SVD 압축을 눈으로

100×100 흑백 이미지(숫자 10,000개)를 만들어 SVD 로 분해한 뒤,
**σ 큰 조각부터 k 개만** 남겨 복원해 봅니다.

$$A \approx \sigma_1 \vec{u}_1\vec{v}_1^{\mathsf T} + \cdots + \sigma_k \vec{u}_k\vec{v}_k^{\mathsf T}$$

rank k 근사에 필요한 저장량은 조각당 $(1 + 100 + 100)$ 개 × k 라서 rank 5면 **1,005개**로
원본의 약 1/10입니다.

In [ ]:
# 간단한 합성 이미지 (구조 + 세부 + 약간의 잡음)
n = 100
yy, xx = np.mgrid[0:n, 0:n]
img = 0.4 * (xx / n)                                        # 좌->우 그라데이션
img += np.exp(-((xx - 32) ** 2 + (yy - 40) ** 2) / 350)     # 밝은 원 1
img += 0.8 * np.exp(-((xx - 70) ** 2 + (yy - 62) ** 2) / 600)  # 밝은 원 2
img[18:24, 12:88] = 1.2                                     # 가로 막대
img[12:88, 76:82] = 1.0                                     # 세로 막대
rng = np.random.default_rng(7)
img += rng.normal(0, 0.03, size=img.shape)                  # 잡음 살짝

def lowrank(M, k):
    """σ 큰 순서로 k 조각만 남긴 rank k 근사."""
    U, S, Vt = np.linalg.svd(M)
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

ranks = [1, 5, 20]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'원본 (숫자 {n*n:,}개)')
for ax, k in zip(axes[1:], ranks):
    ax.imshow(lowrank(img, k), cmap='gray')
    ax.set_title(f'rank {k} (숫자 {k*(2*n+1):,}개)')
for ax in axes:
    ax.axis('off')
plt.suptitle('SVD 저랭크 근사: 조각 몇 개로 이미지가 살아난다', y=1.02)
plt.tight_layout()
plt.show()

rank 5 정도면 큰 구조가 다 보이고, rank 20이면 원본과 거의 구별이 안 됩니다.
게다가 잘 보면 **잡음이 오히려 줄었어요**. 작은 σ 조각에 잡음이 몰려 있었기 때문(노이즈 제거).

## 4. 특이값의 크기 분포: 왜 몇 개만 남겨도 될까

특이값들을 큰 순서로 그려보면, **앞의 몇 개가 정보 대부분을 담고**
나머지는 급격히 작아지는 걸 볼 수 있습니다.

In [ ]:
S_img = np.linalg.svd(img, compute_uv=False)   # 특이값만 계산
energy = np.cumsum(S_img ** 2) / np.sum(S_img ** 2)  # 누적 정보 비율 (ch8 의 λ 비율과 같은 아이디어)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(S_img[:40], 'o-', ms=4, color='#6366f1')
ax1.set_title('특이값 (큰 순서, 앞 40개)')
ax1.set_xlabel('번호'); ax1.set_ylabel('σ')

ax2.plot(energy[:40], 'o-', ms=4, color='#10b981')
ax2.axhline(0.95, color='#ef4444', ls='--', lw=1, label='95% 선')
ax2.set_title('누적 정보 비율')
ax2.set_xlabel('남긴 조각 수 k'); ax2.set_ylim(0, 1.05)
ax2.legend()
plt.tight_layout()
plt.show()

k95 = int(np.argmax(energy >= 0.95)) + 1
print(f'정보의 95% 를 담는 데 필요한 조각 수: {k95}개 (전체 {len(S_img)}개 중)')

---

### 정리

| 개념 | 한 줄 요약 | NumPy |
|------|-----------|-------|
| SVD | 아무 행렬이든 = 회전·확대·회전 | `U, S, Vt = np.linalg.svd(A)` |
| 특이값 σ | 타원 반지름 = 방향별 확대 배율 (내림차순) | `S` |
| rank | 0 아닌 σ 의 개수 (ch9) | `np.linalg.matrix_rank` |
| 저랭크 근사 | σ 큰 조각 k 개만 남긴 최선의 rank k 근사 | `U[:,:k] @ np.diag(S[:k]) @ Vt[:k,:]` |
| σ 곱 | \|det\| = 넓이 배율 (ch6) | `S.prod()` |

**AI 연결**: 추천 시스템(평점 행렬의 취향 축 분해) · 이미지 압축·노이즈 제거 ·
PCA 의 실제 계산법 · 임베딩의 원조 LSA · LoRA(ch9)의 저랭크 통찰.

**다음 챕터(ch12, 마지막)**: 그래디언트와 행렬미분. AI 가 실제로 '학습'하는 원리입니다.